# XGBoost- Classification

In [ ]:
%pip install -r C:\xtra\Last_Chance\git_re\inexorable-ML\requirements.txt

In [ ]:
# importing libs:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import RandomizedSearchCV, cross_val_predict


In [ ]:
df = pd.read_csv("../../../../../data/Travel.csv")
df.head()

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns
non_numeric_cols = df.select_dtypes(exclude='number').columns

In [ ]:
df['Gender'] = df['Gender'].replace('Fe Male', 'Female')
df['Gender'].unique()
df['MaritalStatus'] = df['MaritalStatus'].replace('Single', 'Unmarried')
df['MaritalStatus'].unique()
# reducing unnecessary columns
df['Visited'] = df['NumberOfChildrenVisiting']+df["NumberOfPersonVisiting"]
df.head()

In [ ]:
df = df.drop(columns=['NumberOfChildrenVisiting', 'NumberOfPersonVisiting'])
df.head()
# missing values

df.isnull().sum()[df.isnull().sum() >= 1]


In [ ]:
features_with_na = [
    features for features in df.columns if df[features].isnull().sum()>=1
]
for feature in features_with_na:
    print(feature, np.round(df[feature].isnull().mean()*100,5),"% missing values")

In [ ]:
num_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = df.select_dtypes(include=['object', 'category']).columns.tolist()


discrete_num_features = ['Visited']
continuous_num_features = ['Age', 'DurationOfPitch', 'NumberOfTrips', 'MonthlyIncome']

# Discrete pipeline (mode)
discrete_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('scaler', StandardScaler())
])

# Continuous pipeline (median + scale)
continuous_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # fill missing with mode
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))     # convert to 0/1 vectors
])

preprocessor = ColumnTransformer([
    ('cont', continuous_pipeline, continuous_num_features),
    ('disc', discrete_pipeline, discrete_num_features),
    ('cat', cat_pipeline, cat_features)
])
models = {
    "XGBoost": XGBClassifier(),
}
preprocessor


In [ ]:
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}
pipelines

In [ ]:
pipelines['XGBoost']

In [ ]:
def get_classification_metrics(y_true, y_pred, y_proba=None, model_name=None, verbose=True):
    """
    Calculate standard classification metrics and optionally print them.

    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    y_proba : array-like, optional
        Predicted probabilities for positive class (for ROC AUC)
    model_name : str, optional
        Name of the model for printing
    verbose : bool
        Whether to print the metrics

    Returns:
    --------
    metrics : dict
        Dictionary containing accuracy, precision, recall, f1, roc_auc, confusion_matrix
    """
    metrics = {}
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    metrics['precision'] = precision_score(y_true, y_pred)
    metrics['recall'] = recall_score(y_true, y_pred)
    metrics['f1'] = f1_score(y_true, y_pred)
    if y_proba is not None:
        metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
    else:
        metrics['roc_auc'] = None

    metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred)
    if verbose:
        if model_name:
            print(f"--- {model_name} ---")
        print(f"Accuracy : {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall   : {metrics['recall']:.4f}")
        print(f"F1-score : {metrics['f1']:.4f}")
        print(f"ROC AUC  : {metrics['roc_auc']}")
        print("Confusion Matrix:")
        print(metrics['confusion_matrix'])
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred))
        print("-"*40)
    return metrics


In [ ]:
X = df.drop('ProdTaken', axis=1)
y = df['ProdTaken']

X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.25, random_state=67
    )
for name, cols in [('continuous', continuous_num_features),
                   ('discrete', discrete_num_features),
                   ('categorical', cat_features)]:
    print(f"{name} columns:", cols)

# now transformation:
X_train_transformed = preprocessor.fit_transform(X_train)
X_train_transformed=pd.DataFrame(X_train_transformed)
pd.DataFrame(X_train_transformed)

In [ ]:
X_test_transformed = pd.DataFrame(preprocessor.transform(X_test))
X_test_transformed

In [ ]:
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...")

    # Train the model
    pipe.fit(X_train, y_train)

    # TRAIN metrics
    y_train_pred = pipe.predict(X_train)
    try:
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None
    train_metrics = get_classification_metrics(y_train, y_train_pred, y_train_proba,verbose=True)

    # TEST metrics
    y_test_pred = pipe.predict(X_test)
    try:
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None
    test_metrics = get_classification_metrics(y_test, y_test_pred, y_test_proba,verbose=True)

    results[name] = {'train': train_metrics, 'test': test_metrics}


- **Accuracy:** The proportion of total correct predictions (both classes). On the test set, 88.5% of samples are correctly classified.
- **Precision:** Of all predicted positives, how many were actually positive. Here, ~79.5% of predicted '1's are correct.
- **Recall (Sensitivity):** Of all actual positives, how many were correctly predicted. Only ~45.8% of positive cases were identified → many false negatives.
- **F1-score:** Harmonic mean of precision and recall; balances both. Here, 0.5808 indicates moderate performance on the positive class.
- **ROC AUC:** Probability that a randomly chosen positive ranks higher than a randomly chosen negative. 0.8923 shows good ranking ability despite low recall.
- **Confusion Matrix:** Displays counts of true negatives, false positives, false negatives, and true positives. Shows the model missed 115 positives in the test set.


In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(),
    "AdaBoost": AdaBoostClassifier(),
    "GradBoost": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(),
}
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}
pipelines

In [ ]:
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...")

    # Train the model
    pipe.fit(X_train, y_train)

    # TRAIN metrics
    y_train_pred = pipe.predict(X_train)
    try:
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None
    train_metrics = get_classification_metrics(y_train, y_train_pred, y_train_proba,verbose=True)

    # TEST metrics
    y_test_pred = pipe.predict(X_test)
    try:
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None
    test_metrics = get_classification_metrics(y_test, y_test_pred, y_test_proba,verbose=True)

    results[name] = {'train': train_metrics, 'test': test_metrics}


In [ ]:
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...")

    # Train the model
    pipe.fit(X_train, y_train)

    # TRAIN metrics
    y_train_pred = pipe.predict(X_train)
    try:
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None
    train_metrics = get_classification_metrics(y_train, y_train_pred, y_train_proba,verbose=True)

    # TEST metrics
    y_test_pred = pipe.predict(X_test)
    try:
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None
    test_metrics = get_classification_metrics(y_test, y_test_pred, y_test_proba,verbose=True)

    results[name] = {'train': train_metrics, 'test': test_metrics}


In [ ]:

dt_params = {
    "model__max_depth": [None, 5, 10, 15],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__criterion": ["gini", "entropy"]
}
lr_params = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs", "saga"],
    "model__class_weight": [None, "balanced"]
}
rf_params = {
    "model__n_estimators": [100, 200, 500, 1000],
    "model__max_depth": [None, 5, 8, 10, 15],
    "model__max_features": ["sqrt", 5, 7, 8],
    "model__min_samples_split": [2, 8, 15, 20],
    "model__min_samples_leaf": [1, 2, 4],
    "model__bootstrap": [True, False]
}
svm_params = {
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["linear", "rbf", "poly"],
    "model__gamma": ["scale", "auto"],
    "model__class_weight": [None, "balanced"]
}
ada_params = {
    "model__n_estimators": [50, 100, 200, 500],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
    "model__estimator": [
        DecisionTreeClassifier(max_depth=1),
        DecisionTreeClassifier(max_depth=2),
        DecisionTreeClassifier(max_depth=3)
    ]
}
gb_params = {
    "model__n_estimators": [100, 200, 500, 1000],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],
    "model__max_depth": [3, 5, 7, 10],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__max_features": ["sqrt", "log2", None]
}

xgboost_params = {
    "model__learning_rate" :[0.1,0.01],
    "model__max_depth" : [5,8, 12, 20, 30],
    "model__n_estimators" :[100, 20, 300],
    "model__colsample_bytree" :[0.5,0.8,1,0.3,0.4]
}
randomcv_models = [
    ("Decision Tree",
     DecisionTreeClassifier(),
     dt_params),

    ("Logistic Regression",
     LogisticRegression(max_iter=5000),
     lr_params),

    ("Random Forest",
     RandomForestClassifier(random_state=42),
     rf_params),

    # ("SVM",
    #  SVC(probability=True),
    #  svm_params),

    ("AdaBoost",
     AdaBoostClassifier(random_state=42),
     ada_params),

    ("GradBoost",
     GradientBoostingClassifier(random_state=42),
     gb_params),

    ("XGBoost",
     XGBClassifier(random_state=42),
     xgboost_params),
]

In [ ]:
best_models = {}

for name, model, params in randomcv_models:
    print(f"\n🔎 Tuning {name}...")

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    random_search = RandomizedSearchCV(
        pipe,
        param_distributions=params,
        n_iter=30,
        scoring='f1',
        cv=5,
        verbose=1,
        n_jobs=-1,
        random_state=42
    )

    random_search.fit(X_train, y_train)

    print("Best Params:", random_search.best_params_)
    print("Best CV F1:", random_search.best_score_)

    best_models[name] = random_search.best_estimator_


In [ ]:
results = {}

for name, best_pipe in best_models.items():
    print(f"\n🚀 Training {name} with best params...")

    # Train on full training set
    best_pipe.fit(X_train, y_train)

    # ===== TRAIN METRICS =====
    y_train_pred = best_pipe.predict(X_train)

    try:
        y_train_proba = best_pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None

    train_metrics = get_classification_metrics(
        y_train, y_train_pred, y_train_proba, verbose=True
    )

    # ===== TEST METRICS =====
    y_test_pred = best_pipe.predict(X_test)

    try:
        y_test_proba = best_pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None

    test_metrics = get_classification_metrics(
        y_test, y_test_pred, y_test_proba, verbose=True
    )

    results[name] = {
        "train": train_metrics,
        "test": test_metrics
    }


In [ ]:
results = {}

for name, best_pipe in best_models.items():
    print(f"\n🚀 Training {name} with best params...")

    # Train on full training set
    best_pipe.fit(X_train, y_train)

    # ===== TEST METRICS =====
    y_test_pred = best_pipe.predict(X_test)

    try:
        y_test_proba = best_pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None

    test_metrics = get_classification_metrics(
        y_test, y_test_pred, y_test_proba, verbose=True
    )

    results[name] = {
        "train": train_metrics,
        "test": test_metrics
    }


In [ ]:
# importing libs:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import RandomizedSearchCV, cross_val_predict


In [ ]:
df = pd.read_csv("../../../../../data/Travel.csv")
df.head()

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns
non_numeric_cols = df.select_dtypes(exclude='number').columns

In [ ]:
df['Gender'] = df['Gender'].replace('Fe Male', 'Female')
df['Gender'].unique()
df['MaritalStatus'] = df['MaritalStatus'].replace('Single', 'Unmarried')
df['MaritalStatus'].unique()
# reducing unnecessary columns
df['Visited'] = df['NumberOfChildrenVisiting']+df["NumberOfPersonVisiting"]
df.head()

In [ ]:
df = df.drop(columns=['NumberOfChildrenVisiting', 'NumberOfPersonVisiting'])
df.head()
# missing values

df.isnull().sum()[df.isnull().sum() >= 1]


In [ ]:
features_with_na = [
    features for features in df.columns if df[features].isnull().sum()>=1
]
for feature in features_with_na:
    print(feature, np.round(df[feature].isnull().mean()*100,5),"% missing values")

In [ ]:
num_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = df.select_dtypes(include=['object', 'category']).columns.tolist()


discrete_num_features = ['Visited']
continuous_num_features = ['Age', 'DurationOfPitch', 'NumberOfTrips', 'MonthlyIncome']

# Discrete pipeline (mode)
discrete_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('scaler', StandardScaler())
])

# Continuous pipeline (median + scale)
continuous_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # fill missing with mode
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))     # convert to 0/1 vectors
])

preprocessor = ColumnTransformer([
    ('cont', continuous_pipeline, continuous_num_features),
    ('disc', discrete_pipeline, discrete_num_features),
    ('cat', cat_pipeline, cat_features)
])
models = {
    "XGBoost": XGBClassifier(),
}
preprocessor


In [ ]:
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}
pipelines

In [ ]:
pipelines['XGBoost']

In [ ]:
def get_classification_metrics(y_true, y_pred, y_proba=None, model_name=None, verbose=True):
    """
    Calculate standard classification metrics and optionally print them.

    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    y_proba : array-like, optional
        Predicted probabilities for positive class (for ROC AUC)
    model_name : str, optional
        Name of the model for printing
    verbose : bool
        Whether to print the metrics

    Returns:
    --------
    metrics : dict
        Dictionary containing accuracy, precision, recall, f1, roc_auc, confusion_matrix
    """
    metrics = {}
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    metrics['precision'] = precision_score(y_true, y_pred)
    metrics['recall'] = recall_score(y_true, y_pred)
    metrics['f1'] = f1_score(y_true, y_pred)
    if y_proba is not None:
        metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
    else:
        metrics['roc_auc'] = None

    metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred)
    if verbose:
        if model_name:
            print(f"--- {model_name} ---")
        print(f"Accuracy : {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall   : {metrics['recall']:.4f}")
        print(f"F1-score : {metrics['f1']:.4f}")
        print(f"ROC AUC  : {metrics['roc_auc']}")
        print("Confusion Matrix:")
        print(metrics['confusion_matrix'])
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred))
        print("-"*40)
    return metrics


In [ ]:
X = df.drop('ProdTaken', axis=1)
y = df['ProdTaken']

X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.25, random_state=67
    )
for name, cols in [('continuous', continuous_num_features),
                   ('discrete', discrete_num_features),
                   ('categorical', cat_features)]:
    print(f"{name} columns:", cols)

# now transformation:
X_train_transformed = preprocessor.fit_transform(X_train)
X_train_transformed=pd.DataFrame(X_train_transformed)
pd.DataFrame(X_train_transformed)

In [ ]:
X_test_transformed = pd.DataFrame(preprocessor.transform(X_test))
X_test_transformed

In [ ]:
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...")

    # Train the model
    pipe.fit(X_train, y_train)

    # TRAIN metrics
    y_train_pred = pipe.predict(X_train)
    try:
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None
    train_metrics = get_classification_metrics(y_train, y_train_pred, y_train_proba,verbose=True)

    # TEST metrics
    y_test_pred = pipe.predict(X_test)
    try:
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None
    test_metrics = get_classification_metrics(y_test, y_test_pred, y_test_proba,verbose=True)

    results[name] = {'train': train_metrics, 'test': test_metrics}


- **Accuracy:** The proportion of total correct predictions (both classes). On the test set, 88.5% of samples are correctly classified.
- **Precision:** Of all predicted positives, how many were actually positive. Here, ~79.5% of predicted '1's are correct.
- **Recall (Sensitivity):** Of all actual positives, how many were correctly predicted. Only ~45.8% of positive cases were identified → many false negatives.
- **F1-score:** Harmonic mean of precision and recall; balances both. Here, 0.5808 indicates moderate performance on the positive class.
- **ROC AUC:** Probability that a randomly chosen positive ranks higher than a randomly chosen negative. 0.8923 shows good ranking ability despite low recall.
- **Confusion Matrix:** Displays counts of true negatives, false positives, false negatives, and true positives. Shows the model missed 115 positives in the test set.


In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(),
    "AdaBoost": AdaBoostClassifier(),
    "GradBoost": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(),
}
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}
pipelines

In [ ]:
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...")

    # Train the model
    pipe.fit(X_train, y_train)

    # TRAIN metrics
    y_train_pred = pipe.predict(X_train)
    try:
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None
    train_metrics = get_classification_metrics(y_train, y_train_pred, y_train_proba,verbose=True)

    # TEST metrics
    y_test_pred = pipe.predict(X_test)
    try:
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None
    test_metrics = get_classification_metrics(y_test, y_test_pred, y_test_proba,verbose=True)

    results[name] = {'train': train_metrics, 'test': test_metrics}


In [ ]:
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...")

    # Train the model
    pipe.fit(X_train, y_train)

    # TRAIN metrics
    y_train_pred = pipe.predict(X_train)
    try:
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None
    train_metrics = get_classification_metrics(y_train, y_train_pred, y_train_proba,verbose=True)

    # TEST metrics
    y_test_pred = pipe.predict(X_test)
    try:
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None
    test_metrics = get_classification_metrics(y_test, y_test_pred, y_test_proba,verbose=True)

    results[name] = {'train': train_metrics, 'test': test_metrics}


In [ ]:

dt_params = {
    "model__max_depth": [None, 5, 10, 15],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__criterion": ["gini", "entropy"]
}
lr_params = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs", "saga"],
    "model__class_weight": [None, "balanced"]
}
rf_params = {
    "model__n_estimators": [100, 200, 500, 1000],
    "model__max_depth": [None, 5, 8, 10, 15],
    "model__max_features": ["sqrt", 5, 7, 8],
    "model__min_samples_split": [2, 8, 15, 20],
    "model__min_samples_leaf": [1, 2, 4],
    "model__bootstrap": [True, False]
}
svm_params = {
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["linear", "rbf", "poly"],
    "model__gamma": ["scale", "auto"],
    "model__class_weight": [None, "balanced"]
}
ada_params = {
    "model__n_estimators": [50, 100, 200, 500],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
    "model__estimator": [
        DecisionTreeClassifier(max_depth=1),
        DecisionTreeClassifier(max_depth=2),
        DecisionTreeClassifier(max_depth=3)
    ]
}
gb_params = {
    "model__n_estimators": [100, 200, 500, 1000],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],
    "model__max_depth": [3, 5, 7, 10],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__max_features": ["sqrt", "log2", None]
}

xgboost_params = {
    "model__learning_rate" :[0.1,0.01],
    "model__max_depth" : [5,8, 12, 20, 30],
    "model__n_estimators" :[100, 20, 300],
    "model__colsample_bytree" :[0.5,0.8,1,0.3,0.4]
}
randomcv_models = [
    ("Decision Tree",
     DecisionTreeClassifier(),
     dt_params),

    ("Logistic Regression",
     LogisticRegression(max_iter=5000),
     lr_params),

    ("Random Forest",
     RandomForestClassifier(random_state=42),
     rf_params),

    # ("SVM",
    #  SVC(probability=True),
    #  svm_params),

    ("AdaBoost",
     AdaBoostClassifier(random_state=42),
     ada_params),

    ("GradBoost",
     GradientBoostingClassifier(random_state=42),
     gb_params),

    ("XGBoost",
     XGBClassifier(random_state=42),
     xgboost_params),
]

In [ ]:
best_models = {}

for name, model, params in randomcv_models:
    print(f"\n🔎 Tuning {name}...")

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    random_search = RandomizedSearchCV(
        pipe,
        param_distributions=params,
        n_iter=30,
        scoring='f1',
        cv=5,
        verbose=1,
        n_jobs=-1,
        random_state=42
    )

    random_search.fit(X_train, y_train)

    print("Best Params:", random_search.best_params_)
    print("Best CV F1:", random_search.best_score_)

    best_models[name] = random_search.best_estimator_


In [ ]:
rf_params = {
    "model__n_estimators": [400, 600, 800, 1000,],
    "model__max_depth": [None, 10, 30],
    "model__max_features": ["sqrt", "log2", 5, 6, 7,8],
    "model__min_samples_split": [2, 4, 6, 8],
    "model__min_samples_leaf": [1, 2, 3, 4, 5],
    "model__bootstrap": [True, False],
    "model__criterion": ["gini", "entropy", "log_loss"]
}
lr_params = {
    "model__C": np.logspace(-4, 3, 40),
    "model__solver": ["lbfgs"],
    "model__penalty": ["l2"],
    "model__class_weight": [None, "balanced"]
}
ada_params = {
    "model__n_estimators": [50,70,100,150,200, 300, 400],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0, 1.2],
    # "model__algorithm":['SAMME'],
    "model__estimator": [
        DecisionTreeClassifier(max_depth=1),
        DecisionTreeClassifier(max_depth=2),
        DecisionTreeClassifier(max_depth=3),
        DecisionTreeClassifier(max_depth=4),
    ]
}
gb_params = {
    "model__n_estimators": [100, 200, 500, 1000],        # Number of boosting stages
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3], # Step size shrinkage
    "model__max_depth": [3, 5, 7, 10],                   # Max depth of each tree
    "model__min_samples_split": [2, 5, 10, 20],          # Minimum samples to split a node
    "model__min_samples_leaf": [1, 2, 5, 10],            # Minimum samples at a leaf
    "model__subsample": [0.6, 0.8, 1.0],                # Fraction of samples for each tree
    "model__max_features": ["sqrt", "log2", None],       # Number of features per split
    "model__loss": ["log_loss", "exponential"]           # Loss function
}
xgb_params = {
    "model__n_estimators": [100, 200, 500, 1000],     # Same meaning
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],  # Same meaning
    "model__max_depth": [3, 5, 7, 10],                # Same meaning
    # Closest equivalent to min_samples_split / min_samples_leaf
    "model__min_child_weight": [1, 3, 5, 10],         # Controls minimum leaf instance weight
    # Subsample of rows
    "model__subsample": [0.6, 0.8, 1.0],               # Same concept
    # Equivalent to max_features
    "model__colsample_bytree": [0.6, 0.8, 1.0],        # Fraction of features per tree
    # Regularization (XGBoost-specific, recommended to tune)
    "model__gamma": [0, 0.1, 0.3, 1],                  # Min loss reduction to split
    "model__reg_alpha": [0, 0.1, 1],                   # L1 regularization
    "model__reg_lambda": [1, 5, 10],                   # L2 regularization
    # Objective equivalent to loss
    "model__objective": ["binary:logistic"]            # Equivalent to log_loss
}

random_search = RandomizedSearchCV(
    pipe,
    param_distributions=params,
    n_iter=200,        # larger iteration budget
    scoring='f1_macro',
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)
randomcv_models = [
    ("Logistic Regression",
     LogisticRegression(max_iter=5000),
     lr_params),

    ("Random Forest",
     RandomForestClassifier(class_weight="balanced_subsample"),
     rf_params),

    ("AdaBoost",
     AdaBoostClassifier(random_state=42),
     ada_params),

    ("GradBoost",
     GradientBoostingClassifier(random_state=42),
     gb_params),

    ("XGBoost",
     XGBClassifier(random_state=42),
     xgboost_params),
]

best_models = {}

for name, model, params in randomcv_models:
    print(f"\n🔎 Tuning {name}...")

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    random_search = RandomizedSearchCV(
        pipe,
        param_distributions=params,
        n_iter=75,
        scoring='f1_macro',
        cv=3,
        verbose=2,
        n_jobs=-1,
        random_state=42
    )


    random_search.fit(X_train, y_train)

    print("Best Params:", random_search.best_params_)
    print("Best CV F1:", random_search.best_score_)

    best_models[name] = random_search.best_estimator_


In [ ]:
results = {}

for name, best_pipe in best_models.items():
    print(f"\n🚀 Training {name} with best params...")

    # Train on full training set
    best_pipe.fit(X_train, y_train)

    # ===== TRAIN METRICS =====
    y_train_pred = best_pipe.predict(X_train)

    try:
        y_train_proba = best_pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None

    train_metrics = get_classification_metrics(
        y_train, y_train_pred, y_train_proba, verbose=True
    )

    # ===== TEST METRICS =====
    y_test_pred = best_pipe.predict(X_test)

    try:
        y_test_proba = best_pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None

    test_metrics = get_classification_metrics(
        y_test, y_test_pred, y_test_proba, verbose=True
    )

    results[name] = {
        "train": train_metrics,
        "test": test_metrics
    }


In [ ]:
results = {}

for name, best_pipe in best_models.items():
    print(f"\n🚀 Training {name} with best params...")

    # Train on full training set
    best_pipe.fit(X_train, y_train)

    # ===== TEST METRICS =====
    y_test_pred = best_pipe.predict(X_test)

    try:
        y_test_proba = best_pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None

    test_metrics = get_classification_metrics(
        y_test, y_test_pred, y_test_proba, verbose=True
    )

    results[name] = {
        "train": train_metrics,
        "test": test_metrics
    }


In [ ]:
# Example: AdaBoost threshold tuning
pipe = best_models["XGBoost"]

y_train_proba_cv = cross_val_predict(
    pipe,
    X_train,
    y_train,
    cv=5,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

thresholds = np.linspace(0.05, 0.95, 91)
f1_scores = []

for t in thresholds:
    y_pred = (y_train_proba_cv >= t).astype(int)
    f1_scores.append(f1_score(y_train, y_pred))

best_threshold = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)

print(f"Best threshold: {best_threshold:.2f}")
print(f"CV F1-score  : {best_f1:.4f}")

# Apply threshold to test set
y_test_proba = pipe.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= best_threshold).astype(int)

get_classification_metrics(y_test, y_test_pred, y_test_proba, verbose=True)


In [ ]:
# Define models
models_to_eval = {
    "Random Forest": best_models["Random Forest"],
    "AdaBoost": best_models["AdaBoost"],
    "GradBoost": best_models["GradBoost"],
    "XGBoost": best_models["XGBoost"]
}

results_thresholds = {}

plt.figure(figsize=(8, 6))

for name, pipe in models_to_eval.items():
    # CV probabilities for threshold tuning
    y_train_proba_cv = cross_val_predict(
        pipe, X_train, y_train, cv=5, method="predict_proba", n_jobs=-1
    )[:, 1]

    # Find F1-optimal threshold
    thresholds = np.linspace(0.05, 0.95, 91)
    f1_scores = [f1_score(y_train, (y_train_proba_cv >= t).astype(int)) for t in thresholds]
    best_threshold = thresholds[np.argmax(f1_scores)]
    best_f1 = max(f1_scores)

    # Predict on test set using best threshold
    y_test_proba = pipe.predict_proba(X_test)[:, 1]
    y_test_pred_f1 = (y_test_proba >= best_threshold).astype(int)

    # ROC curve & AUC
    fpr, tpr, roc_thresholds = roc_curve(y_test, y_test_proba)
    auc_score = roc_auc_score(y_test, y_test_proba)

    # Youden's J for ROC-based threshold
    j_scores = tpr - fpr
    roc_threshold = roc_thresholds[np.argmax(j_scores)]

    # Plot ROC
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_score:.3f})")

    # Save results
    results_thresholds[name] = {
        "best_f1_threshold": best_threshold,
        "best_f1_score": best_f1,
        "roc_threshold": roc_threshold,
        "auc": auc_score,
        "y_test_pred_f1": y_test_pred_f1
    }

# Random baseline
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.grid(alpha=0.3)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# Print thresholds
for name, info in results_thresholds.items():
    print(f"\n{name}:")
    print(f" - Best F1 threshold: {info['best_f1_threshold']:.3f} (CV F1 = {info['best_f1_score']:.4f})")
    print(f" - ROC-based threshold: {info['roc_threshold']:.3f} (AUC = {info['auc']:.3f})")


In [ ]:
# Example for Random Forest with F1-optimal threshold:
best_rf_threshold = 0.430

y_test_proba_rf = best_models["Random Forest"].predict_proba(X_test)[:, 1]
y_test_pred_rf = (y_test_proba_rf >= best_rf_threshold).astype(int)

# Similarly for AdaBoost:
best_ada_threshold = 0.480

y_test_proba_ada = best_models["AdaBoost"].predict_proba(X_test)[:, 1]
y_test_pred_ada = (y_test_proba_ada >= best_ada_threshold).astype(int)


# Similarly for GradBoost:
best_ada_threshold = 0.480

y_test_proba_ada = best_models["GradBoost"].predict_proba(X_test)[:, 1]
y_test_pred_ada = (y_test_proba_ada >= best_ada_threshold).astype(int)
